# Week 3 Day 1 Lab - AI Solutions Architect

## Setting up AWS for Week 3

From Day 2, UnderCover runs on AWS: the app on Lambda, and the agent on Bedrock AgentCore with its Gateway, Policy, Memory and Runtime.

Today we set up the account they run in: the Paid plan, a budget that emails you, and an IAM user with only the permissions the course needs. Then we check each step from Python.

## 6 steps:

STEP 1: Open an AWS account

STEP 2: Turn on MFA for root

STEP 3: Choose us-east-1

STEP 4: Billing access and a budget

STEP 5: Create the IAM user

STEP 6: Install the AWS CLI

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">For the AWS route you need an AWS account on the Paid plan. The Free plan can't use Bedrock AgentCore. The Paid plan needs a card, and new customers get up to $200 of credits. Nothing in today's lab costs anything, but always watch the billing screens and alerts.<br/><br/>
            If you'd rather not use AWS, you have two other options. You can use Render instead, which is free: the free version of this lab, <a href="week3day1free.ipynb">week3day1free.ipynb</a>, sets up your Render account, and each day this week has written instructions for Render. Or you can watch me do it all on AWS.<br/><br/>
            Steps 1 to 5 happen in the AWS console in your browser, and Step 6 in a terminal. After that, run the cells to check each step.
            </span>
        </td>
    </tr>
</table>

In [ ]:
from pathlib import Path

import boto3

## Step 1: Open an AWS account

Go to [aws.amazon.com](https://aws.amazon.com) and choose **Create an AWS account**. You'll need an email address, a card and a phone number. When AWS asks you to choose a plan, choose the **Paid plan**.

The account is usually ready within minutes, but activation can take up to 24 hours.

If you already have an AWS account, you can use it. If it's on the Free plan, upgrade it to the Paid plan in the **Billing and Cost Management** console. Your remaining credits still apply after the upgrade.

The email address and password you signed up with belong to the **root user**, which can do anything in the account, including closing it. We use the root user today to set the account up. From then on we work as an IAM user with fewer permissions.

## Step 2: Turn on MFA for root

Anyone with the root user's password controls the account, so give it a second factor. Signed in as root, open the account menu at the top right of the console, choose **Security credentials**, then **Assign MFA device**. A passkey or an authenticator app on your phone both work.

## Step 3: Choose us-east-1

The region selector is at the top right of the console. Choose **US East (N. Virginia) us-east-1** and leave it there.

We build everything this week in us-east-1, wherever you are. Every AgentCore feature we use is available there, including Memory, which six regions don't have. Using one region also keeps your console the same as mine.

Most of what we create belongs to one region. If you can't find something in the console, check the region selector first.

## Step 4: Billing access and a budget

There are two things to do as root.

**Billing access for IAM users.** Open the account menu, choose **Account** and scroll to **IAM user and role access to Billing information**. Choose **Edit**, tick **Activate IAM Access** and choose **Update**. Without this, an IAM user can't read budgets or costs, whatever its policy allows.

**A budget.** Open **Billing and Cost Management**, choose **Budgets**, then **Create budget**. Choose **Customize (advanced)** and **Cost budget**, and set a monthly budget of $10. Under **Budget scope**, add a **Charge type** filter that excludes **Credit** and **Refund**. Then add alert thresholds at 50%, 80% and 100% of actual costs, each one emailing you.

Credits need excluding because a cost budget counts them by default. While your credits cover the bill, a budget that counts them sees a cost of zero and never alerts you.

## Step 5: Create the IAM user

Still as root, open **IAM** (search for it at the top of the console).

1. **The policy.** Choose **Policies**, **Create policy** and the **JSON** tab. Replace the contents with the policy in `labs/undercover-course-policy.json`, which the next cell prints. Choose **Next**, name it `undercover-course` and create it.
2. **The group.** Choose **User groups** and **Create group**. Name it `undercover-course` and attach the `undercover-course` policy.
3. **The user.** Choose **Users** and **Create user**. Call it `undercover` and add it to the `undercover-course` group. We use this user from the terminal and from code, so it doesn't need console access.
4. **An access key.** Open the user, choose **Security credentials** and **Create access key**, and pick **Command Line Interface (CLI)**. AWS suggests alternatives first, so tick the confirmation and continue. Copy the access key and the secret access key, or download the .csv file. AWS shows the secret only once.

Treat the access key like a password. Keep it out of notebooks and out of git.

If you already have an IAM user from my Production Track course, add it to the `undercover-course` group and carry on with it.

The policy allows what we do on AWS this week and little else: AgentCore, the CloudFormation stacks and CDK resources that AgentCore's CLI deploys with, Lambda functions and IAM roles named `undercover...`, the encryption keys and secrets for our API keys, logs, and reading the account's plan, budgets and costs. If a step later in the week fails with `AccessDenied`, the message names the action that the policy is missing.

In [ ]:
print(Path("undercover-course-policy.json").read_text(encoding="utf-8"))

## Step 6: Install the AWS CLI

In a terminal:

- PC: `winget install -e --id Amazon.AWSCLI`
- Mac: download the installer from https://awscli.amazonaws.com/AWSCLIV2.pkg and run it

Quit VS Code and open it again so the notebook and VS Code's terminal find `aws`, and open a new one if you use a terminal outside VS Code. Then check it with `aws --version`.

Then run `aws configure` and give it the access key and the secret access key from Step 5, `us-east-1` as the default region and `json` as the output format. It saves them in a `.aws` folder in your home folder, outside the repo, where the AWS CLI and Python's AWS library both find them. Newer versions of the CLI then ask whether to configure AWS skills and the AWS MCP server for your AI coding agents. Answer `n` for this course, so your coding agent's setup doesn't change partway through.

The next two cells run the CLI from the notebook. The second is the check from AWS's own instructions, and it prints your account and the user you're signed in as.

In [ ]:
!aws --version

In [ ]:
!aws sts get-caller-identity

## Checking each step

Now we check the steps from Python with boto3, AWS's Python library. It reads the key and the region that `aws configure` saved.

### Who you are (Step 5)

In [ ]:
session = boto3.Session()
identity = session.client("sts").get_caller_identity()

print(f"Account: {identity['Account']}")
print(f"Signed in as: {identity['Arn']}")
if identity["Arn"].endswith(":root"):
    print("These are the root user's keys. Create an access key for your IAM user in Step 5 and run aws configure again.")

### The region (Step 6)

This is the region you gave `aws configure`. The console's region selector from Step 3 is a setting in your browser, so we can't check it from here.

In [ ]:
print(f"Region: {session.region_name}")
if session.region_name != "us-east-1":
    print("Run aws configure again and set the default region to us-east-1")

### The Paid plan (Step 1)

In [ ]:
plan = session.client("freetier").get_account_plan_state()
print(f"Plan: {plan['accountPlanType']} ({plan['accountPlanStatus']})")

remaining = plan.get("accountPlanRemainingCredits")
if remaining:
    print(f"Credits left: ${remaining['amount']:,.2f}")
if plan["accountPlanType"] != "PAID":
    print("AgentCore needs the Paid plan. Upgrade the account in Billing and Cost Management, then run this cell again.")

### MFA on root (Step 2)

In [ ]:
summary = session.client("iam").get_account_summary()["SummaryMap"]

if summary["AccountMFAEnabled"]:
    print("The root user has MFA")
else:
    print("The root user has no MFA yet. Go back to Step 2.")

### The budget (Step 4)

A budget made in today's console keeps its filters in `FilterExpression`, and an older one keeps its charge types in `CostTypes`. This function reads either to see whether credits count against the budget.

In [ ]:
def counts_credits(budget):
    """Whether a cost budget counts AWS credits against its limit."""
    excluded = budget.get("FilterExpression", {}).get("Not", {}).get("Dimensions", {})
    if excluded.get("Key") == "RECORD_TYPE":
        return "Credit" not in excluded["Values"]
    return budget.get("CostTypes", {}).get("IncludeCredit", True)

If the next cell fails with `AccessDeniedException`, go back to Step 4 and activate IAM access to billing.

In [ ]:
budgets = session.client("budgets").describe_budgets(AccountId=identity["Account"]).get("Budgets", [])

for budget in budgets:
    limit = float(budget["BudgetLimit"]["Amount"])
    print(f"{budget['BudgetName']}: ${limit:,.2f} {budget['TimeUnit'].lower()}")
    if counts_credits(budget):
        print("  This budget counts credits, so it won't alert you while credits cover the bill. Add the Charge type filter from Step 4.")
    else:
        print("  Credits excluded")

if not budgets:
    print("No budget yet. Create one in Step 4.")

### AgentCore (Step 5)

The last check asks AgentCore for its agent runtimes. An empty list means the policy is attached and AgentCore is answering. We create nothing on AgentCore until Day 2.

In [ ]:
runtimes = session.client("bedrock-agentcore-control").list_agent_runtimes()
print(runtimes["agentRuntimes"])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Costs from Day 2</h2>
            <span style="color:#118AB2;">Today's setup costs nothing. From Day 2 we create resources that cost a few cents a day while they exist, so each day's lab ends with a teardown that deletes them. The app on Lambda costs nothing while nobody uses it, so it stays up until the final teardown on Day 5. The budget's emails are there in case anything gets left behind.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">At a client you'll usually work in their cloud account, with the permissions their IT team grants you. A policy like <code>undercover-course</code>, listing what your project needs, is the kind of document their security team will want to review before they give you access. A budget with alerts and MFA on the root user are the usual first safeguards on any account.<br/><br/>
            Consider what you would need to ask for to deploy an agent in your own organization's cloud.
            </span>
        </td>
    </tr>
</table>